# Constructing and Testing Scraper Pipelines

Use this notebook to _create_, _test_, and _validate_ each particular scraping function (for each data source). The final functions will be added as methods to the _EventScraper_ class.

In [10]:
# import required libraries
import requests
import pandas as pd

# import scraping packages and libraries
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from bs4 import BeautifulSoup
from scrapy import Selector
import pyperclip

# import supporting packages for system interactions
import random
import time
import os
import re

In [2]:
# create helper function to use selenium for dynamically rendered pages
def fetch_rendered_html(url: str, wait_for_selector: str = None, timeout: int = 15) -> BeautifulSoup:
    """
    Fetches a URL using headless Chrome, waits for JS execution,
    and returns a BeautifulSoup instance of the fully rendered DOM.
    """
    # 1. Configure Chrome Options for Headless Scraping
    options = Options()
    options.add_argument("--headless=new")  # Modern headless mode
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # Spoof realistic User-Agent to avoid bot blocks
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    # 2. Initialize Driver
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(url)

        # 3. Handle JavaScript Wait Conditions
        if wait_for_selector:
            # Explicit Wait: Pauses until target dynamic element appears in the DOM
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, wait_for_selector))
            )
        else:
            # Fallback: Brief sleep for general JS script execution
            time.sleep(3)

        # 4. Extract the fully rendered DOM
        rendered_html = driver.page_source
        return BeautifulSoup(rendered_html, "html.parser")

    finally:
        # Always close the browser instance to prevent memory leaks
        driver.quit()

# use the US census API to get latitude and longitude
def get_census_coordinates(address_string):
    """
    Takes a full address string (e.g. "1600 Pennsylvania Ave NW, Washington, DC 20006")
    and returns (latitude, longitude) or (None, None) if not found.
    """
    if not address_string or pd.isna(address_string):
        return None, None
        
    url = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"
    params = {
        "address": address_string,
        "benchmark": "Public_AR_Current",
        "format": "json"
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        matches = data.get("result", {}).get("addressMatches", [])
        if matches:
            # Extract coordinates (Census API returns x=longitude, y=latitude)
            coordinates = matches[0]["coordinates"]
            return coordinates["y"], coordinates["x"]  # Latitude, Longitude
    except Exception as e:
        print(f"Geocoding failed for '{address_string}': {e}")
        
    return None, None

## Tacoma Dome

---

In [3]:
# set the head url as the link value
link = "https://www.tacomadome.org/events"

# set the venue location as the default location for the events
location = "2727 E D St, Tacoma, WA 98421"
loc_link = "https://maps.app.goo.gl/bT8bE5ERX2Jdbybp8" # in the future we will use Google Maps API to get the location link, but for now we will hardcode it

# get the HTML content of the page
response = requests.get(link)
soup = BeautifulSoup(response.content, 'html.parser')

# filter to only the section of interest, which is the table containing the data we want to extract
target_table = soup.find("div", class_="full_column non-widget-area") # IMPORTANT: this is the div that contains the table we want to extract

# strip out unnecessary scripts and styles
for script in target_table(["script", "style"]):
    script.decompose()

# find all blocks where the class has the pattern r"eventItem
pat = re.compile(r"^eventItem.*clearfix$")

# find all divs with the class that matches the pattern
event_items = target_table.find_all(class_=pat)

# create a dictionary to hold the event data
event_dict = {}
td_df = pd.DataFrame()

# parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
for event in event_items:
    # get the name of the event
    try:
        event_dict["title"] = event.find("h3").get_text(strip=True)
    except AttributeError:
        event_dict["title"] = None

    # get the link to the details page
    try:
        event_dict["external_url"] = event.find("a")["href"]
    except AttributeError:
        event_dict["external_url"] = None

    # get the date range of the event
    try:
        event_dict["start_time"] = event.find("div", class_="date").get("aria-label").strip()
    except AttributeError:
        event_dict["start_time"] = None

    # get the description of the event
    try:
        event_dict["description"] = event.find("h4").get_text(strip=True)
    except AttributeError:
        event_dict["description"] = None

    # get the image link of the event
    try:
        event_dict["image_url"] = event.find("img")["src"]
    except AttributeError:
        event_dict["image_url"] = None
        
    # get the ticket link of the event (if available)
    try:
        event_dict["ticket_link"] = event.find("a", class_="tickets onsalenow").get("href").strip()
    except AttributeError:
        event_dict["ticket_link"] = None

    # concatenate the event_dict to the dataframe
    td_df = pd.concat([td_df, pd.DataFrame([event_dict])], ignore_index=True)

# add the location and location link to the dataframe (invariant for all events at the Tacoma Dome)
td_df["address"] = location
td_df["location_link"] = loc_link
td_df["venue_name"] = "Tacoma Dome"

# reset the index
td_df = td_df.reset_index(drop=True)

## Emerald Queen

---

In [ ]:
# set the head url as the link value
link = "https://emeraldqueen.com/tickets/"

# set the venue location as the default location for the events
location = "2920 E R St, Tacoma, WA 98404"
loc_link = "https://maps.app.goo.gl/oLm111Ag9aezpN458" # in the future we will use Google Maps API to get the location link, but for now we will hardcode it

# get the HTML content of the page
response = requests.get(link)
soup = BeautifulSoup(response.content, 'html.parser')

# filter to only the section of interest, which is the table containing the data we want to extract
target_table = soup.find("section", class_="content page-925 moto-section") # IMPORTANT: this is the div that contains the table we want to extract

# find all divs with the class that matches the pattern
event_items = target_table.find_all("div", class_="moto-widget moto-widget-row moto-spacing-top-medium moto-spacing-right-auto moto-spacing-bottom-medium moto-spacing-left-auto")

# create a dictionary to hold the event data
event_dict = {}
eqc_df = pd.DataFrame()

# parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
for event in event_items:
    # batch extract primary details
    try:
        details = event.find_all("div", class_="moto-widget-text-content moto-widget-text-editable")
        # extract the name, date range, and description from the details
        event_dict["title"] = details[0].get_text(strip=True)
        event_dict["start_time"] = details[1].get_text(strip=True)
        event_dict["description"] = details[2].get_text(strip=True)
    except:
        event_dict["title"] = None
        event_dict["start_time"] = None
        event_dict["description"] = None

    # get the link to the details page
    try:
        event_dict["external_url"] = "https://emeraldqueen.com" + event.find("a")["href"]
    except:
        event_dict["external_url"] = None

    # get the image link of the event
    try:
        event_dict["image_url"] = "https://emeraldqueen.com" + event.find("img").get("data-src")
    except:
        event_dict["image_url"] = None

    # follow the details link to get richer details
    details_response = requests.get(event_dict["external_url"])
    details_soup = BeautifulSoup(details_response.content, 'html.parser')
        
    # get the ticket link of the event (if available)
    try:
        event_dict["ticket_link"] = details_soup.find("a", href=re.compile(r"ticketmaster", re.IGNORECASE)).get("href")
    except:
        event_dict["ticket_link"] = None

    # concatenate the event_dict to the dataframe
    eqc_df = pd.concat([eqc_df, pd.DataFrame([event_dict])], ignore_index=True)

# add the location and location link to the dataframe (invariant for all events at the Tacoma Dome)
eqc_df["address"] = location
eqc_df["location_link"] = loc_link
eqc_df["venue_name"] = "Emerald Queen Casino"

# reset the index
eqc_df = eqc_df.reset_index(drop=True)

## Puyallup Fairgrounds

---

In [6]:
# set the link value
link = "https://www.thefair.com/events-calendar/"

# set the venue location as the default location for the events
location = "110 9th Ave SW, Puyallup, WA 98371"
loc_link = "https://maps.app.goo.gl/SobVsYk1KdtJjR5s6" # in the future we will use Google Maps API to get the location link, but for now we will hardcode it

# create a final dataframe to hold all pages' worth of data
final_df = pd.DataFrame()

# iterate through the pages of the events calendar and extract the event data
for page in range(1, 10): # adjust the range as needed to scrape more pages
    # get html from a dynamically rendered page
    try:
        soup = fetch_rendered_html(url=link + f"?page={page}")
    except Exception as e:
        print(f"Error fetching page {page}: {e}")
        break  # Stop the loop if there's an error fetching the page

    # filter to only the section of interest, which is the table containing the data we want to extract
    target_table = soup.find("div", class_="container container--md2") # IMPORTANT: this is the div that contains the table we want to extract

    # find all divs with the class that matches the pattern
    event_items = target_table.find_all("div", class_="event-card-large")
    # create a dictionary to hold the event data
    event_dict = {}
    puy_df = pd.DataFrame()

    # parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
    for event in event_items:
        # get name of event
        name_info = event.find("h3")
        try:
            event_dict["title"] = name_info.get_text(strip=True)
        except:
            event_dict["title"] = None

        # get the date range of the event
        try:
            event_dict["start_time"] = event.find("div", class_="icon-text").get_text(strip=True)
        except:
            event_dict["start_time"] = None

        # get the link to the details page
        try:
            event_dict["external_url"] = "https://www.thefair.com/events" + name_info.find("a")["href"]
        except:
            event_dict["external_url"] = None

        # get the image link of the event
        try:
            event_dict["image_url"] = "https://www.thefair.com/events" + event.find("img").get("src")
        except:
            event_dict["image_url"] = None
            
        # get the ticket link of the event (if available)
        try:
            event_dict["ticket_link"] = event.find("div", class_="event-card-large__buttons").find("a", href=re.compile(r"ticket")).get("href").strip()
        except:
            try:
                detail_soup = fetch_rendered_html(url=event_dict["details_link"])
                event_dict["ticket_link"] = detail_soup.find("a", href=re.compile(r"ticket")).get("href").strip()
            except:
                event_dict["ticket_link"] = None

        # get the event description
        try:
            event_dict["description"] = event.find("span").get_text(strip=True)
        except:
            event_dict["description"] = None

        # concatenate the event_dict to the dataframe
        puy_df = pd.concat([puy_df, pd.DataFrame([event_dict])], ignore_index=True)
        
    # concatenate the intermediate dataframe to the final
    final_df = pd.concat([final_df, puy_df], ignore_index=True)

# add the location and location link to the dataframe (invariant for all events at the Tacoma Dome)
final_df["address"] = location
final_df["location_link"] = loc_link
final_df["venue_name"] = "Puyallup Fairgrounds"

# reset the index
final_df = final_df.reset_index(drop=True)

# convert the 'start_time' to a standard datetime obj w/ formatting
final_df["start_time"] = pd.to_datetime(final_df["start_time"], errors='coerce')

# sort by name and date, ascending
final_df = final_df.sort_values(by=["title", "start_time"])

# group by name, taking first for all but start_time, then taking min-max dates from start_time
final_df = final_df.groupby("title").agg({
    "start_time": lambda x : f"{x.min().strftime('%b %d')} - {x.max().strftime('%b %d %Y')}" if x.min() != x.max() else x.min().strftime('%b %d %Y'),
    "external_url": "first",
    "image_url": "first",
    "ticket_link": "first",
    "description": "first",
    "address": "first",
    "location_link": "first",
    "venue_name": "first"
}).reset_index()

# Add Final Details to the Table

In [7]:
# concatenate all tables produced so far
final = pd.concat([td_df, eqc_df, final_df]).reset_index(drop=True)

In [8]:
# use US census API to get lat/ long from address
# ================================================
# first get only unique addresses
loc_info = pd.DataFrame({
    "address":sorted(set(final["address"]))
})

# get lat/ long corresponding to the unique addresses
loc_info["coordinates"] = loc_info["address"].apply(get_census_coordinates)

# create lat and long columns from coords, then drop coords
loc_info["latitude"] = loc_info["coordinates"].apply(lambda x : x[0])
loc_info["longitude"] = loc_info["coordinates"].apply(lambda x : x[1])
loc_info = loc_info.drop(columns=["coordinates"])

# join the small table back to add in lat/ long data
final = pd.merge(
    left=final,
    right=loc_info,
    on="address",
    how="left"
)

In [ ]:
# strip out all apostrophe/ single quote chars from the entire table

In [ ]:
# format the table so that each row is in the literal string form
sql_table = ""

for i, row in final.iterrows():
    sql_table = sql_table + f"""
            (
            '{final.loc[i, "title"]}',
            '{final.loc[i, "venue_name"]}',
            '{final.loc[i, "address"]}',
            {final.loc[i, "latitude"]},
            {final.loc[i, "longitude"]},
            '{final.loc[i, "description"]}',
            NULL,
            '{final.loc[i, "start_time"]}',
            {random.random()},
            {random.random()},
            {random.random()},
            NULL,
            '{final.loc[i, "external_url"]}',
            '{final.loc[i, "image_url"]}'
            ),
        """

In [17]:
print(sql_table)


            (
            'Tacoma Fall RV Show',
            'Tacoma Dome',
            '2727 E D St, Tacoma, WA 98421',
            47.237434510215,
            -122.428675061895,
            '350+ RV's Inside the Tacoma Dome & 100+ Used RV's Outside',
            NULL,
            'October  1 to October  4 2026',
            0.05105804477356468,
            0.28879925344559176,
            0.25630592323250245,
            NULL,
            'https://www.tacomadome.org/events/detail/tacoma-fall-rv-show-2026'
            'https://www.tacomadome.org/assets/img/events_placeholder-87c3faa39f.png'
            ),
        
            (
            'Disney Descendants, ZOMBIES & Camp Rock: Worlds Collide Concert Tour',
            'Tacoma Dome',
            '2727 E D St, Tacoma, WA 98421',
            47.237434510215,
            -122.428675061895,
            'None',
            NULL,
            'October 10 2026',
            0.44606177261554425,
            0.03809249203697418,
          